# 03 — Classical baseline: does combining features beat `abs_asym` alone?

**Revalidation (2026-09-08):** the first pass of this notebook loaded
`data/processed/eda_features.csv`, computed by section 6d's inline EDA
code -- the *uncorrected* striatum mask (no central-region restriction).
`src/features.py` now has the corrected version (graduated with tests,
see `README.md`), so this notebook was rewritten to **recompute
`abs_asym`/`striatal_ratio` for all 1362 volumes using the real
`src/features.py` code** rather than reuse the stale CSV -- validates the
code that will actually ship, not a notebook prototype. The original
run's win margin was large (-0.1054 vs. baseline) and the mask bug
plausibly affected only ~2-6% of volumes, so the result was expected to
hold, but that was an assumption, not a check.

Minimal-scope baseline (per `structuring-ml-projects` SKILL.md step 6,
the gate): `LogisticRegression` on `abs_asym` + `striatal_ratio`,
compared against each alone and against `config.BASELINE_LOGLOSS`, using
`evaluate.make_folds`/`log_loss_score` consistently across all three
(apples-to-apples).

**Decision this feeds (the gate, SKILL.md step 6):** if `combined`
doesn't beat `abs_asym` alone by more than the noise floor (this run's
own per-feature-set sd), `striatal_ratio` is a negative result (SKILL.md
step 7) — log it, don't keep it.

**Data handling:** the recompute cell reads `.nii.gz` files, so it is
marked **[RUN ME]** — run it yourself, share back only the printed
aggregate output. It also writes `data/processed/baseline_features.csv`
(derived scalar features + label, not pixel data) for reuse.

In [ ]:
# [RUN ME] loads pixel data -- recomputes features via the real, graduated
# src/features.py code (not the old inline EDA logic) for all 1362 volumes.
import sys
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
import config
import features

TARGET_ML = 20.0


def inplane_family(v):
    """Same family buckets as EDA section 3a, so folds stay comparable."""
    for lo, hi, name in [(1.40, 1.50, "~1.47"), (1.50, 1.80, "~1.5-1.8"),
                         (1.99, 2.01, "2.00"), (2.29, 2.31, "2.30"),
                         (2.39, 2.41, "2.398"), (2.45, 2.47, "2.46"),
                         (3.28, 3.32, "~3.30"), (3.58, 3.60, "3.591"),
                         (3.88, 3.90, "3.895"), (4.41, 4.43, "4.42")]:
        if lo <= v < hi:
            return name
    return f"other({v:.3f})"


labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)

rows = []
skipped = []
for uid, label in zip(labels_df[config.UID_COLUMN], labels_df[config.TARGET_COLUMN]):
    img = nib.load(str(config.NIFTI_DIR / f"{uid}.nii.gz"))
    volume = img.get_fdata()
    spacing = img.header.get_zooms()[:3]

    mask = features.striatum_mask(volume, spacing, target_ml=TARGET_ML)
    if mask is None:
        skipped.append(uid)
        continue
    signed = features.signed_asymmetry(volume, mask, spacing)
    ratio = features.striatal_ratio(volume, mask)
    if signed is None:
        skipped.append(uid)
        continue

    rows.append({
        config.UID_COLUMN: uid,
        config.TARGET_COLUMN: label,
        "inplane_family": inplane_family(spacing[0]),
        "abs_asym": abs(signed),
        "signed_asym": signed,
        "striatal_ratio": ratio,
    })

feat_v2 = pd.DataFrame(rows)
print(f"{len(feat_v2)} volumes processed via src/features.py, "
      f"{len(skipped)} skipped (degenerate mask)")
if skipped:
    print("skipped uids (no labels shown):", skipped)

out_path = config.DATA_PROCESSED / "baseline_features.csv"
feat_v2.to_csv(out_path, index=False)
print("Saved to", out_path)


In [ ]:
# [RUN ME] -- pure pandas/sklearn on the CSV just written, no new pixel access.
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import config
import evaluate

feat_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")
y = feat_df[config.TARGET_COLUMN].to_numpy()
family = feat_df["inplane_family"].to_numpy()

N_SEEDS = 5
feature_sets = {
    "abs_asym": ["abs_asym"],
    "striatal_ratio": ["striatal_ratio"],
    "combined": ["abs_asym", "striatal_ratio"],
}

results = {}
header = "feature set"
print(f"{header:<16} {'mean':>8} {'sd':>8} {'vs baseline':>14}")
for name, cols in feature_sets.items():
    X = feat_df[cols].to_numpy(dtype=float)
    seed_scores = []
    for seed in range(N_SEEDS):
        folds = evaluate.make_folds(y, family, n_splits=config.N_FOLDS, random_state=seed)
        preds = np.zeros(len(y))
        for train_idx, test_idx in folds:
            model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
            model.fit(X[train_idx], y[train_idx])
            preds[test_idx] = model.predict_proba(X[test_idx])[:, 1]
        seed_scores.append(evaluate.log_loss_score(y, preds))
    seed_scores = np.array(seed_scores)
    results[name] = seed_scores
    print(f"{name:<16} {seed_scores.mean():>8.4f} {seed_scores.std():>8.4f} "
          f"{seed_scores.mean() - config.BASELINE_LOGLOSS:>+14.4f}")

combined_vs_abs = results["combined"].mean() - results["abs_asym"].mean()
combined_vs_ratio = results["combined"].mean() - results["striatal_ratio"].mean()
print(f"\ncombined vs. abs_asym alone: delta = {combined_vs_abs:+.4f}")
print(f"combined vs. striatal_ratio alone: delta = {combined_vs_ratio:+.4f}")


**What we're looking for:** whether the gate decision from the first
(uncorrected-mask) pass survives recomputation with the real, graduated
`src/features.py` code.

**Why:** `SKILL.md` step 6 — only wire in what wins, validated against
the actual code, not a prototype that later changed.

**Source:** `structuring-ml-projects` SKILL.md step 6.

**What we found** (run 2026-09-08, `src/features.py` with the
central-region-restricted mask, `TARGET_ML=20.0`, 5 seeds × `config.N_FOLDS`):

- **1362/1362 volumes processed, 0 skipped** (degenerate mask) — the
  central-region restriction didn't cost any volumes relative to the
  original 6a/6b pass.

| feature set | mean | sd | vs. baseline |
|---|---|---|---|
| abs_asym | 0.5889 | 0.0003 | -0.0996 |
| striatal_ratio | 0.6647 | 0.0003 | -0.0238 |
| **combined** | **0.5753** | **0.0006** | **-0.1132** |

- combined vs. abs_asym alone: delta = **-0.0136**
- combined vs. striatal_ratio alone: delta = -0.0894

The noise floor (per-feature-set seed sd) is 0.0003-0.0006. The
combined-vs-abs_asym margin (-0.0136) is ~20-45× that — far outside
noise, not a coin flip. This also reproduces the *direction and rough
magnitude* of the first (uncorrected-mask) pass's -0.1054 combined-vs-baseline
result (now -0.1132), so the central-region fix did not change the
qualitative conclusion, only the exact numbers.

**Decision / next step:** the gate is **passed** — `combined` beats
`abs_asym` alone by more than the noise floor, using the real
`src/features.py` code on all 1362 volumes. `striatal_ratio` is confirmed
as a genuine (not spurious/prototype-only) contributor, not a negative
result. Keep both `abs_asym` and `striatal_ratio` in
`src/model.py::build_classical_baseline()`; no further revalidation of
this specific gate is needed unless `src/features.py`'s mask logic
changes again.